# EDA — Customer Churn

Laboratorio de Minería de Datos · ISTEA · Proyecto Integrador

Exploración del dataset histórico antes de entrenar nada. La idea es entender
qué hay, qué falta y qué problemas de calidad tiene, porque esas decisiones
después condicionan el pipeline.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/raw/customer_churn_historical.csv")
df.shape

## 1. Dimensiones y tipos

In [ ]:
df.head()

In [ ]:
df.info()

Acá aparece el primer problema de calidad: `TotalCharges` figura como
`object` (texto) cuando debería ser numérica. Veamos por qué.

In [ ]:
# Cuantas filas no se pueden convertir a numero
no_numericas = pd.to_numeric(df["TotalCharges"], errors="coerce").isna().sum()
print("Valores que no son numericos en TotalCharges:", no_numericas)

# Como se ven esas filas
df[pd.to_numeric(df["TotalCharges"], errors="coerce").isna()].head()

**Decisión:** se convierte con `errors="coerce"`, que deja esos valores como
`NaN`, y la imputación se hace dentro del pipeline. No se edita el CSV a mano:
eso arreglaría un archivo, no el proceso.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dtypes

## 2. Valores faltantes

In [ ]:
faltantes = df.isna().sum()
faltantes[faltantes > 0]

## 3. Distribución del target

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="Churn", ax=ax)
ax.set_title("Distribucion de la variable objetivo")
plt.show()

**Dato importante:** las clases están desbalanceadas. Eso tiene dos
consecuencias directas:

1. Hay que usar `stratify` en la partición, para que train y test mantengan la
   misma proporción.
2. **Accuracy no sirve como métrica principal.** Un modelo que prediga siempre
   "No" ya acierta el porcentaje de la clase mayoritaria sin haber aprendido
   nada. Por eso el baseline del entrenamiento es un `DummyClassifier`: para
   tener ese piso a la vista.

## 4. Variables numéricas

In [ ]:
numericas = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
df[numericas].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.histplot(data=df, x=col, hue="Churn", bins=30, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

Mirar acá si hay alguna variable donde las dos clases se separen visualmente.
`tenure` suele ser la más marcada en este dataset: los clientes con poca
antigüedad se van más.

## 5. Variables categóricas

In [ ]:
categoricas = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]

for col in categoricas:
    print(col, "->", df[col].nunique(), "categorias")

In [ ]:
# Tasa de churn por categoria, para las que suelen ser mas relevantes
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    tasa = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean())
    print("\n--- Tasa de churn por", col, "---")
    print(tasa.round(3).sort_values(ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
tasa_contract = df.groupby("Contract")["Churn"].apply(lambda s: (s == "Yes").mean())
tasa_contract.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Tasa de churn segun modalidad contractual")
ax.set_xlabel("proporcion de clientes que abandonan")
plt.tight_layout()
plt.show()

## 6. Problemas de calidad detectados

In [ ]:
# Duplicados
print("Filas duplicadas:", df.duplicated().sum())
print("customerID duplicados:", df["customerID"].duplicated().sum())

# Rangos raros
print("\ntenure minimo:", df["tenure"].min(), "| maximo:", df["tenure"].max())
print("MonthlyCharges minimo:", df["MonthlyCharges"].min(),
      "| maximo:", df["MonthlyCharges"].max())

## 7. El umbral de decisión

El modelo devuelve una probabilidad. Para convertirla en "se va / no se va"
hace falta un corte, y 0.50 es solo el valor por defecto.

La consigna pide discutir qué impacto tiene un falso negativo: decir que un
cliente se queda cuando en realidad se estaba yendo. Ese error es más caro que
el falso positivo, porque el cliente se pierde y ni siquiera se intentó
retenerlo. Un falso positivo, en cambio, solo cuesta la promoción.

In [ ]:
import joblib
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Se rehace la particion igual que en src/data/load.py
X = df.drop(columns=["Churn", "customerID"])
y = (df["Churn"] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

pipeline = joblib.load("../models/churn_pipeline.joblib")
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("Impacto del umbral")
print("umbral  precision  recall   f1     falsos_neg  falsos_pos")
for u in np.arange(0.20, 0.75, 0.05):
    y_pred = (y_proba >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    print("%.2f    %.3f      %.3f    %.3f   %-11d %d" % (
        u,
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        fn, fp))

Bajar el umbral detecta más clientes en riesgo (menos falsos negativos) pero
genera más falsas alarmas.

_(Completar: qué umbral se elige y por qué. Después hay que cambiarlo en
`DECISION_THRESHOLD` dentro de `src/config.py`.)_

### Resumen del EDA

_(Completar con las conclusiones propias después de correr las celdas.)_

- **Dimensiones:** ... filas × ... columnas
- **Faltantes:** ... en `TotalCharges`, tratados por imputación dentro del pipeline
- **Balance del target:** ... % de churn
- **Variables que más separan:** ...
- **Decisiones que se llevan al pipeline:** ...

**`customerID` no se usa como predictor**: es un identificador, y el README del
dataset lo pide explícitamente.